In [1]:
import pandas as pd
import logging

# Configure basic logging to display in Jupyter
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True
)

logging.info("Libraries imported and logger configured successfully.")

2026-06-01 21:47:39,614 - INFO - Libraries imported and logger configured successfully.


In [2]:
input_file = 'raw_sensor_data.csv'

try:
    df = pd.read_csv(input_file)
    logging.info(f"Loaded raw data from '{input_file}'. Initial shape: {df.shape[0]} rows, {df.shape[1]} columns.")
except FileNotFoundError:
    logging.error(f"Could not find '{input_file}'. Please ensure it is in the same folder as this notebook.")
    raise

2026-06-01 21:48:35,340 - INFO - Loaded raw data from 'raw_sensor_data.csv'. Initial shape: 1646 rows, 4 columns.


In [3]:
# 1. Drop exact duplicate rows
initial_rows = len(df)
df_clean = df.drop_duplicates()
logging.info(f"Dropped {initial_rows - len(df_clean)} duplicate rows.")

# 2. Drop rows missing critical metadata
current_rows = len(df_clean)
df_clean = df_clean.dropna(subset=['sensor_id', 'timestamp'])
logging.info(f"Dropped {current_rows - len(df_clean)} rows missing a 'sensor_id' or 'timestamp'.")

2026-06-01 21:48:57,292 - INFO - Dropped 46 duplicate rows.
2026-06-01 21:48:57,304 - INFO - Dropped 24 rows missing a 'sensor_id' or 'timestamp'.


In [4]:
# 3. Impute missing values with median substitution
df_clean['value'] = df_clean.groupby('sensor_type')['value'].transform(lambda x: x.fillna(x.median()))

logging.info("Imputed missing sensor values with median substitution based on sensor type.")

2026-06-01 21:49:10,405 - INFO - Imputed missing sensor values with median substitution based on sensor type.


In [5]:
# 4. Filter outliers based on realistic bounds
valid_ranges = {
    'temperature': {'min': -20, 'max': 60},       
    'humidity': {'min': 0, 'max': 100},           
    'pressure': {'min': 900, 'max': 1100},        
    'light_lux': {'min': 0, 'max': 2000},         
    'vibration_hz': {'min': 0, 'max': 100}        
}

def filter_outliers(row):
    s_type = row['sensor_type']
    val = row['value']
    if s_type in valid_ranges:
        if valid_ranges[s_type]['min'] <= val <= valid_ranges[s_type]['max']:
            return True
        else:
            return False
    return True # Keep it if we don't have rules for that specific sensor

rows_before_outliers = len(df_clean)
mask = df_clean.apply(filter_outliers, axis=1)
df_clean = df_clean[mask]

logging.info(f"Dropped {rows_before_outliers - len(df_clean)} outlier rows outside valid ranges.")

2026-06-01 21:49:21,929 - INFO - Dropped 38 outlier rows outside valid ranges.


In [6]:
# 5. Format timestamp
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean = df_clean.sort_values(by='timestamp')

# 6. Export to CSV
output_file = 'cleaned_sensor_data.csv'
df_clean.to_csv(output_file, index=False)

logging.info(f"Pipeline complete! Saved cleaned data to '{output_file}'. Final shape: {df_clean.shape[0]} rows.")

# Display the first 5 rows to verify it looks pristine
df_clean.head()

2026-06-01 21:49:40,391 - INFO - Pipeline complete! Saved cleaned data to 'cleaned_sensor_data.csv'. Final shape: 1538 rows.


,timestamp,sensor_id,sensor_type,value
0,2023-10-01 00:02:00+00:00,S_006,light_lux,622.56
1,2023-10-01 00:07:00+00:00,S_005,pressure,1004.73
2,2023-10-01 00:09:00+00:00,S_001,temperature,17.90
3,2023-10-01 00:14:00+00:00,S_002,temperature,29.57
4,2023-10-01 00:15:00+00:00,S_001,temperature,17.74
